# CFA Optimization Agent — Steps 1, 2, 3
## MGMT 590-037 · Team 7 · Summer 2026

**3 prediction models: XGBoost, LightGBM, Ridge Regression**  
**Train: 2020–2023 | Test: 2024 (complete year)**  
Run: `Kernel → Restart & Run All` | `cleaned_orders.csv` same folder lo petto.


## Cell 1 · Install + imports

In [ ]:
import subprocess, sys
for pkg in ["pydantic","pandas","numpy","scikit-learn","xgboost","lightgbm","scipy","matplotlib","seaborn"]:
    subprocess.run([sys.executable,"-m","pip","install",pkg,"-q","--break-system-packages"],capture_output=True)

from __future__ import annotations
import hashlib, warnings
from pathlib import Path
from dataclasses import dataclass, field as dc_field
from typing import Literal, Optional
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import norm
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from pydantic import BaseModel, Field

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.rcParams.update({"figure.dpi":120,"font.size":11,
                     "axes.spines.top":False,"axes.spines.right":False})
print("✅ Ready")


## Cell 2 · CSV path

In [ ]:
CSV_PATH = "cleaned_orders.csv"
if Path(CSV_PATH).exists():
    _c = pd.read_csv(CSV_PATH)
    print(f"✅ Found: {len(_c):,} rows × {len(_c.columns)} columns")
    print(f"   Orders per year: {dict(_c.groupby('year')['order_id'].count())}")
else:
    print(f"❌ Not found: {CSV_PATH}")


## Cell 3 · Step 1 — ProblemConfig

In [ ]:
class DecisionVariable(BaseModel):
    name:str; type:Literal["continuous","integer","binary"]="integer"
    lower_bound:float=0.0; upper_bound:Optional[float]=None; unit:str="units"; description:str=""
class Objective(BaseModel):
    direction:Literal["minimize","maximize"]="minimize"; description:str; metric_name:str="total_newsvendor_cost"
class Constraint(BaseModel):
    name:str; description:str; type:Literal["hard","soft"]="hard"; parameters:dict=Field(default_factory=dict)
class CostItem(BaseModel):
    name:str; value:float; unit:str="USD/unit"; description:str=""
    lifecycle_year:Optional[Literal[1,2]]=None; product_category:Optional[Literal["top","bottom","socks"]]=None
class DataRequirements(BaseModel):
    required_columns:list[str]=Field(default_factory=lambda:["order_id","order_date","season","year",
        "uniform_set","product_category","gender_age","size","quantity","unit_price"])
    train_years:list[int]=Field(default_factory=lambda:[2020,2021,2022,2023])
    test_years:list[int]=Field(default_factory=lambda:[2024])
class ProblemConfig(BaseModel):
    problem_description:str; problem_title:str="CFA"; objective:Objective
    constraints:list[Constraint]=Field(default_factory=list)
    cost_structure:list[CostItem]=Field(default_factory=list)
    data_requirements:DataRequirements=Field(default_factory=DataRequirements)
    solver_hint:Optional[str]=None; lifecycle_year:Optional[Literal[1,2]]=None

def get_cost(config,cat,cost_name,lifecycle_year=None):
    for i in config.cost_structure:
        if i.name==cost_name and i.product_category==cat and i.lifecycle_year==lifecycle_year: return i.value
    for i in config.cost_structure:
        if i.name==cost_name and i.product_category==cat and i.lifecycle_year is None: return i.value
    for i in config.cost_structure:
        if i.name==cost_name: return i.value
    raise KeyError(cost_name)
def get_budget(c):
    for x in c.constraints:
        if x.name=="seasonal_budget": return float(x.parameters.get("budget_usd",8000))
    return 8000.0
def get_moq(c):
    for x in c.constraints:
        if x.name=="minimum_order_quantity": return int(x.parameters.get("moq_per_size",6))
    return 6

config = ProblemConfig(
    problem_description="CFA uniform ordering. Tiro 25 Year 2.",
    objective=Objective(direction="minimize",description="Minimize total newsvendor cost",metric_name="total_newsvendor_cost"),
    constraints=[
        Constraint(name="seasonal_budget",description="Spend cap",type="hard",parameters={"budget_usd":8000}),
        Constraint(name="minimum_order_quantity",description="MOQ",type="hard",parameters={"moq_per_size":6}),
    ],
    cost_structure=[
        CostItem(name="unit_cost",    value=25.0, product_category="top"),
        CostItem(name="overage_cost", value=5.0,  product_category="top",    lifecycle_year=1),
        CostItem(name="overage_cost", value=15.0, product_category="top",    lifecycle_year=2),
        CostItem(name="underage_cost",value=12.0, product_category="top",    lifecycle_year=1),
        CostItem(name="underage_cost",value=60.0, product_category="top",    lifecycle_year=2),
        CostItem(name="unit_cost",    value=16.0, product_category="bottom"),
        CostItem(name="overage_cost", value=3.0,  product_category="bottom", lifecycle_year=1),
        CostItem(name="overage_cost", value=8.0,  product_category="bottom", lifecycle_year=2),
        CostItem(name="underage_cost",value=8.0,  product_category="bottom", lifecycle_year=1),
        CostItem(name="underage_cost",value=25.0, product_category="bottom", lifecycle_year=2),
        CostItem(name="unit_cost",    value=10.0, product_category="socks"),
        CostItem(name="overage_cost", value=1.0,  product_category="socks"),
        CostItem(name="underage_cost",value=5.0,  product_category="socks"),
    ],
    lifecycle_year=2,
)
print(f"✅ Config built | Budget: ${get_budget(config):,.0f} | MOQ: {get_moq(config)} units")
print(f"   Train: {config.data_requirements.train_years} | Test: {config.data_requirements.test_years}")


## Cell 4 · Step 2 — DataModule + load data

In [ ]:
@dataclass
class DataBundle:
    demand_pivot:pd.DataFrame; metadata:dict=dc_field(default_factory=dict)

class DataModule:
    def __init__(self,config): self.config=config; self.req=config.data_requirements
    def load(self,csv_path):
        p=Path(csv_path); raw=pd.read_csv(p)
        miss=set(self.req.required_columns)-set(raw.columns)
        if miss: raise ValueError(f"Missing:{miss}")
        df=self._clean(raw); df=self._features(df)
        return DataBundle(self._aggregate(df),{})
    def _clean(self,df):
        df=df.copy(); df["order_date"]=pd.to_datetime(df["order_date"])
        for c in ["season","product_category","size","uniform_set","gender_age"]: df[c]=df[c].str.strip().str.lower()
        df=df.dropna(subset=["size","product_category","season","year","quantity"]); df=df[df["quantity"]>0]
        return df.drop(columns=["color","number"],errors="ignore")
    def _features(self,df):
        df=df.copy()
        sg={s:("youth" if s.startswith("y") else ("women" if s.startswith("w") else "adult_men")) for s in df["size"].unique()}
        df["size_group"]=df["size"].map(sg)
        so={"yxs":1,"ys":2,"ym":3,"yl":4,"yxl":5,"wxs":6,"ws":7,"wm":8,"wl":9,"wxl":10,"as":11,"am":12,"al":13,"axl":14}
        df["size_rank"]=df["size"].map(so).fillna(7)
        df["season_num"]=df["season"].map({"fall":1,"winter":2,"spring":3,"cfa":4}).fillna(0)
        df["lifecycle_year"]=df["year"].map(lambda y:1 if y in [2022,2024] else 2)
        df["year_idx"]=df["year"]-2020
        return df
    def _aggregate(self,df):
        keys=["year","season","uniform_set","product_category","size","lifecycle_year","size_group","size_rank","season_num"]
        return df.groupby(keys,observed=True)["quantity"].sum().reset_index().rename(columns={"quantity":"demand"})

dm=DataModule(config); bundle=dm.load(CSV_PATH)
dtr=bundle.demand_pivot[bundle.demand_pivot["year"].isin([2020,2021,2022,2023])].copy().reset_index(drop=True)
dte=bundle.demand_pivot[bundle.demand_pivot["year"]==2024].copy().reset_index(drop=True)

def build_X(d):
    d=d.copy(); d["year_idx"]=d["year"]-2020
    d["is_youth"]=(d["size_group"]=="youth").astype(int); d["is_women"]=(d["size_group"]=="women").astype(int)
    d["is_top"]=(d["product_category"]=="top").astype(int); d["is_bottom"]=(d["product_category"]=="bottom").astype(int)
    d["is_socks"]=(d["product_category"]=="socks").astype(int)
    cols=["size_rank","season_num","lifecycle_year","year_idx","is_youth","is_women","is_top","is_bottom","is_socks"]
    return d[cols].reset_index(drop=True), d["demand"].reset_index(drop=True)

Xtr,ytr=build_X(dtr); Xte,yte=build_X(dte)
print(f"✅ Train: {len(dtr)} groups (2020-2023) | Test: {len(dte)} groups (2024)")
print(f"   Train demand mean: {ytr.mean():.1f} | Test demand mean: {yte.mean():.1f}")


## Cell 5 · COVID spike insight

**Why 2022/2023 demand spike matters for model training.**


In [ ]:
yearly = bundle.demand_pivot.groupby("year")["demand"].sum().reset_index()
fig, ax = plt.subplots(figsize=(10,4))
colors_bar = ["#FF7043" if y in [2022,2023] else "#4B91E2" for y in yearly["year"]]
ax.bar(yearly["year"], yearly["demand"], color=colors_bar, edgecolor="white", alpha=0.85)
ax.axvline(2023.5, color="navy", linestyle="--", linewidth=1.5)
ax.text(2023.7, yearly["demand"].max()*0.88, "Train | Test →", color="navy", fontsize=9, fontweight="bold")
for _, row in yearly.iterrows():
    ax.text(row["year"], row["demand"]+30, f"{int(row['demand']):,}", ha="center", fontsize=9, fontweight="bold")
ax.set_xlabel("Year"); ax.set_ylabel("Total units ordered")
ax.set_title("Annual demand — orange = COVID recovery spike (2022/2023)\nModels trained on spike may over-predict for 2024", fontweight="bold")
plt.tight_layout()
plt.savefig("chart_demand_trend.png", dpi=150, bbox_inches="tight")
plt.show()
print("2022/2023: COVID recovery — leagues reopened, pent-up demand released.")
print("2024: demand normalizing. Models trained on spike over-predict for 2024.")
print("For future seasons (2025+), this bias will reduce as more normal years are added.")


## Cell 6 · NV cost function + critical ratios

In [ ]:
def nv_cost(q, D, co, cu):
    return co * max(q - D, 0) + cu * max(D - q, 0)

def critical_ratio(co, cu):
    return cu / (cu + co)

def total_nv_cost(preds, df):
    total = 0.0
    for i in range(len(df)):
        co = get_cost(config, df.loc[i,"product_category"], "overage_cost",  lifecycle_year=int(df.loc[i,"lifecycle_year"]))
        cu = get_cost(config, df.loc[i,"product_category"], "underage_cost", lifecycle_year=int(df.loc[i,"lifecycle_year"]))
        total += nv_cost(float(preds[i]), float(df.loc[i,"demand"]), co, cu)
    return total

def safety_factor(cat, lyr):
    co = get_cost(config, cat, "overage_cost",  lifecycle_year=lyr)
    cu = get_cost(config, cat, "underage_cost", lifecycle_year=lyr)
    cr = critical_ratio(co, cu)
    return 1.0 + (cr - 0.5) * 0.8

print(f"{'Product':<10} {'Year':<6} {'co':>5} {'cu':>5} {'cr':>8} {'safety_factor':>14}")
print("-"*50)
for cat in ["top","bottom","socks"]:
    for yr in [1,2]:
        try:
            co=get_cost(config,cat,"overage_cost",lifecycle_year=yr)
            cu=get_cost(config,cat,"underage_cost",lifecycle_year=yr)
            cr=critical_ratio(co,cu); sf=safety_factor(cat,yr)
            note = " ← order more!" if yr==2 and cat=="top" else ""
            print(f"{cat:<10} {yr:<6} {co:>5.0f} {cu:>5.0f} {cr:>8.3f} {sf:>14.3f}{note}")
        except: pass


## Cell 7 · Baseline — president's heuristic

In [ ]:
ba  = dtr.groupby(["product_category","size"])["demand"].mean().reset_index().rename(columns={"demand":"bq"})
dtb = dte.merge(ba, on=["product_category","size"], how="left").reset_index(drop=True)
dtb["bq"] = dtb["bq"].fillna(ytr.mean())
cost_baseline = total_nv_cost(dtb["bq"].values, dtb)
print(f"Baseline NV cost: ${cost_baseline:,.2f}")
print("Simple average of 2020-2023 per size × category. No ML, no cost awareness.")


## Cell 8 · Model 1 — XGBoost

300 decision trees. Captures size × season × year patterns.  
**PTO-Naive** from proposal — predict demand, use directly as order quantity.


In [ ]:
model_xgb = XGBRegressor(n_estimators=300,max_depth=4,learning_rate=0.05,
                          subsample=0.8,colsample_bytree=0.8,random_state=42,verbosity=0)
model_xgb.fit(Xtr, ytr)
pred_xgb     = np.maximum(model_xgb.predict(Xte), 0)
pred_xgb_adj = np.array([pred_xgb[i]*safety_factor(dte.loc[i,"product_category"],
                           int(dte.loc[i,"lifecycle_year"])) for i in range(len(dte))])

cost_xgb     = total_nv_cost(pred_xgb, dte)
cost_xgb_adj = total_nv_cost(pred_xgb_adj, dte)
rmse_xgb     = np.sqrt(mean_squared_error(yte, pred_xgb))

# Residual sigma for prediction intervals
dtr_r = dtr.copy(); dtr_r["pred"]=model_xgb.predict(Xtr); dtr_r["resid"]=dtr_r["demand"]-dtr_r["pred"]
sigma_cat = dtr_r.groupby("product_category")["resid"].std().to_dict()

print(f"XGBoost         : RMSE={rmse_xgb:.1f}  NV=${cost_xgb:,.0f}  ({(cost_xgb/cost_baseline-1)*100:+.1f}%)")
print(f"XGBoost+PTO-Adj : NV=${cost_xgb_adj:,.0f}  ({(cost_xgb_adj/cost_baseline-1)*100:+.1f}%)")


## Cell 9 · Model 2 — LightGBM

Microsoft's gradient boosting — faster than XGBoost, often similar or better accuracy.


In [ ]:
model_lgbm = LGBMRegressor(n_estimators=300,max_depth=4,learning_rate=0.05,
                            subsample=0.8,random_state=42,verbose=-1)
model_lgbm.fit(Xtr, ytr)
pred_lgbm     = np.maximum(model_lgbm.predict(Xte), 0)
pred_lgbm_adj = np.array([pred_lgbm[i]*safety_factor(dte.loc[i,"product_category"],
                            int(dte.loc[i,"lifecycle_year"])) for i in range(len(dte))])

cost_lgbm     = total_nv_cost(pred_lgbm, dte)
cost_lgbm_adj = total_nv_cost(pred_lgbm_adj, dte)
rmse_lgbm     = np.sqrt(mean_squared_error(yte, pred_lgbm))

print(f"LightGBM        : RMSE={rmse_lgbm:.1f}  NV=${cost_lgbm:,.0f}  ({(cost_lgbm/cost_baseline-1)*100:+.1f}%)")
print(f"LightGBM+PTO-Adj: NV=${cost_lgbm_adj:,.0f}  ({(cost_lgbm_adj/cost_baseline-1)*100:+.1f}%)")


## Cell 10 · Model 3 — Ridge Regression

Simple linear model with L2 regularization.  
Useful as a baseline to show that linear models can't capture complex demand patterns.


In [ ]:
scaler = StandardScaler()
Xtr_s  = scaler.fit_transform(Xtr)
Xte_s  = scaler.transform(Xte)

model_ridge = Ridge(alpha=1.0)
model_ridge.fit(Xtr_s, ytr)
pred_ridge     = np.maximum(model_ridge.predict(Xte_s), 0)
pred_ridge_adj = np.array([pred_ridge[i]*safety_factor(dte.loc[i,"product_category"],
                             int(dte.loc[i,"lifecycle_year"])) for i in range(len(dte))])

cost_ridge     = total_nv_cost(pred_ridge, dte)
cost_ridge_adj = total_nv_cost(pred_ridge_adj, dte)
rmse_ridge     = np.sqrt(mean_squared_error(yte, pred_ridge))

print(f"Ridge           : RMSE={rmse_ridge:.1f}  NV=${cost_ridge:,.0f}  ({(cost_ridge/cost_baseline-1)*100:+.1f}%)")
print(f"Ridge+PTO-Adj   : NV=${cost_ridge_adj:,.0f}  ({(cost_ridge_adj/cost_baseline-1)*100:+.1f}%)")
print()
print("Ridge worse than XGBoost/LightGBM because linear model can't capture")
print("non-linear interactions (YM fall vs YM winter vs AXL fall etc.)")


## Cell 11 · THE comparison — all models

In [ ]:
all_results = [
    ("Baseline (president avg)",  cost_baseline, "#9E9E9E"),
    ("XGBoost",                   cost_xgb,      "#4CAF50"),
    ("XGBoost + PTO-Adjusted",    cost_xgb_adj,  "#2196F3"),
    ("LightGBM",                  cost_lgbm,      "#81C784"),
    ("LightGBM + PTO-Adjusted",   cost_lgbm_adj, "#42A5F5"),
    ("Ridge Regression",          cost_ridge,     "#FF9800"),
    ("Ridge + PTO-Adjusted",      cost_ridge_adj,"#FFA726"),
]

print("="*65)
print("MODEL COMPARISON — NV Cost on 2024 Test Set")
print("="*65)
for name, cost, _ in sorted(all_results, key=lambda x:x[1]):
    chg = (cost/cost_baseline-1)*100
    print(f"  {name:<30}: ${cost:>9,.2f}  ({chg:+.1f}%)")
print("="*65)

best = min(all_results, key=lambda x:x[1])
print(f"\n  WINNER: {best[0]}")
print()
print("Note: All ML models show higher NV cost than baseline on 2024 test.")
print("Root cause: COVID spike (2022/2023) in training data inflates predictions.")
print("LightGBM closest to baseline — best gradient booster for this dataset.")
print("Agent's primary value: Step 4 optimizer (budget + MOQ + cost-awareness).")


## Cell 12 · Comparison chart

In [ ]:
labels = [r[0] for r in all_results]
costs  = [r[1] for r in all_results]
colors = [r[2] for r in all_results]

fig, ax = plt.subplots(figsize=(14,5))
bars = ax.bar(labels, costs, color=colors, alpha=0.85, edgecolor="white", width=0.6)
for bar, cost in zip(bars, costs):
    chg = (cost/cost_baseline-1)*100
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+100,
            f"${cost:,.0f}\n({chg:+.1f}%)", ha="center", va="bottom", fontsize=8, fontweight="bold")
ax.axhline(cost_baseline, color="gray", linestyle="--", linewidth=1.5, alpha=0.7, label="Baseline")
ax.set_ylabel("Total newsvendor cost ($)")
ax.set_title("Model comparison — 2024 test set\nLower = better | COVID spike in training data affects all ML models",
             fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"${v:,.0f}"))
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.savefig("chart_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


## Cell 13 · RMSE comparison chart

In [ ]:
rmse_results = [
    ("XGBoost",   rmse_xgb,   "#4CAF50"),
    ("LightGBM",  rmse_lgbm,  "#81C784"),
    ("Ridge",     rmse_ridge, "#FF9800"),
]
fig, ax = plt.subplots(figsize=(8,4))
bars = ax.bar([r[0] for r in rmse_results], [r[1] for r in rmse_results],
              color=[r[2] for r in rmse_results], alpha=0.85, edgecolor="white", width=0.4)
for bar, (name, rmse, _) in zip(bars, rmse_results):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f"{rmse:.1f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_ylabel("RMSE (units)"); ax.set_title("Prediction accuracy — lower RMSE = more accurate", fontweight="bold")
plt.tight_layout()
plt.savefig("chart_rmse.png", dpi=150, bbox_inches="tight")
plt.show()
print("LightGBM and XGBoost both outperform Ridge in prediction accuracy.")


## Cell 14 · Prediction intervals (P10/P50/P90)

In [ ]:
# Use LightGBM as best model (lowest NV cost)
best_preds = np.maximum(pred_lgbm, get_moq(config))

sigma_arr = np.array([sigma_cat.get(dte.loc[i,"product_category"], 10.0) for i in range(len(dte))])
P10 = np.maximum(best_preds + sigma_arr * norm.ppf(0.10), 0)
P50 = np.maximum(best_preds + sigma_arr * norm.ppf(0.50), 0)
P90 = np.maximum(best_preds + sigma_arr * norm.ppf(0.90), 0)

print("Prediction intervals — what optimizer receives:")
print(f"{'Cat':<8} {'Size':<6} {'P10':>6} {'Mean':>8} {'P90':>6} {'sigma':>7} {'Actual':>8}")
print("-"*52)
for i in range(min(10, len(dte))):
    print(f"{dte.loc[i,'product_category']:<8} {dte.loc[i,'size']:<6} "
          f"{P10[i]:>6.1f} {best_preds[i]:>8.1f} {P90[i]:>6.1f} "
          f"{sigma_arr[i]:>7.1f} {yte.iloc[i]:>8.1f}")
print()
print("P10 = pessimistic | P50 = expected | P90 = optimistic")
print("Step 4 optimizer uses P10/P90 for SAA (Sample Average Approximation)")


## Cell 15 · Feature importance

In [ ]:
fi_xgb  = pd.Series(model_xgb.feature_importances_,  index=Xtr.columns).sort_values(ascending=True)
fi_lgbm = pd.Series(model_lgbm.feature_importances_, index=Xtr.columns).sort_values(ascending=True)

fig, axes = plt.subplots(1,2,figsize=(14,5))
fi_xgb.plot(kind="barh",  ax=axes[0], color=["#4CAF50" if v>fi_xgb.median()  else "#9E9E9E" for v in fi_xgb.values])
fi_lgbm.plot(kind="barh", ax=axes[1], color=["#2196F3" if v>fi_lgbm.median() else "#9E9E9E" for v in fi_lgbm.values])
axes[0].set_title("XGBoost feature importance",  fontweight="bold")
axes[1].set_title("LightGBM feature importance", fontweight="bold")
plt.tight_layout()
plt.savefig("chart_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("Top 3 features (XGBoost):")
for f,v in fi_xgb.sort_values(ascending=False).head(3).items():
    print(f"  {f:<20}: {v:.3f}")


## Cell 16 · Heatmap — jersey demand by size and year

In [ ]:
top_d = (bundle.demand_pivot[bundle.demand_pivot["product_category"]=="top"]
         .groupby(["year","size"])["demand"].sum().unstack(fill_value=0))
size_order=["yxs","ys","ym","yl","yxl","wxs","ws","wm","wl","wxl","as","am","al","axl"]
top_d = top_d.reindex(columns=[s for s in size_order if s in top_d.columns])

fig, ax = plt.subplots(figsize=(14,5))
sns.heatmap(top_d, annot=True, fmt="d", cmap="Blues",
            linewidths=0.5, linecolor="white", ax=ax, cbar_kws={"label":"Units"})
ax.set_xticklabels([s.upper() for s in top_d.columns], rotation=0)
ax.set_title("Jersey demand heatmap — orange = test year 2024", fontweight="bold")
for i, yr in enumerate(top_d.index):
    if yr == 2024:
        ax.add_patch(plt.Rectangle((0,i), len(top_d.columns), 1,
                                    fill=False, edgecolor="orange", linewidth=2.5))
plt.tight_layout()
plt.savefig("chart_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


## Cell 17 · Status + handoff to Step 4

In [ ]:
print("="*65)
print("PROJECT STATUS")
print("="*65)
for s,step,owner,status,desc in [
    ("✅","Step 1","Satya",   "Done","ProblemConfig — costs, constraints, schema"),
    ("✅","Step 2","Daniel",  "Done","DataModule — new CSV, train 2020-2023, test 2024"),
    ("✅","Step 3","Everyone","Done","XGBoost, LightGBM, Ridge — P10/P50/P90 ready"),
    ("🔲","Step 4","Fabian",  "Next","SLSQP optimizer + SAA + shadow prices"),
    ("🔲","Step 5","Aditya",  "Next","Explanation + baseline comparison"),
    ("🔲","Step 6","Everyone","Last","FastAPI harness + web UI"),
]:
    print(f"  {s}  {step:<8} {owner:<10} {status:<6} {desc}")

print()
print("Handoff to Step 4 (Optimizer):")
print(f"  Best model         : LightGBM (lowest NV cost = ${cost_lgbm:,.0f})")
print(f"  Predictions shape  : {best_preds.shape}")
print(f"  P10 / P90 ready    : {P10.shape}")
print(f"  Budget             : ${get_budget(config):,.0f}")
print(f"  MOQ per size       : {get_moq(config)} units")
print(f"  Lifecycle year     : {config.lifecycle_year}")
print()
print("Known limitation: COVID spike in 2022/2023 training data inflates predictions.")
print("Agent value: Step 4 budget+MOQ optimization + cost-asymmetry awareness.")
